In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import PowerTransformer,LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
import joblib
from sklearn.metrics import classification_report


In [ ]:
train=pd.read_csv(r"D:\4projects\Instagram_Account_Authentication\data\Insta_train.csv")
test=pd.read_csv(r"D:\4projects\Instagram_Account_Authentication\data\Insta_test.csv")


In [6]:
train.drop_duplicates(inplace=True)
train.rename(columns={'#posts': 'posts', '#followers': 'followers', '#follows': 'follows'},inplace=True)
test.rename(columns={'#posts': 'posts', '#followers': 'followers', '#follows': 'follows'},inplace=True)

In [7]:
cat_col=['profile pic','external URL','name==username','private']

for col in cat_col:
    train[col] = train[col].astype(bool)
    test[col] = test[col].astype(bool)


In [8]:
def add_features(df):
    df['follower_follow_ratio'] = df['followers'] / (df['follows'] + 1)
    df['follow_post_ratio'] = df['follows'] / (df['posts'] + 1)
    df['SuspicionScore'] = (
        (~df['external URL']).astype(int) +
        (~df['profile pic']).astype(int) +
        (df['name==username']).astype(int) +
        (df['posts'] == 0).astype(int) +
        (df['followers'] < 50).astype(int) +
        ((df['follower_follow_ratio'] < 0.1) & (df['followers'] > 1000)).astype(int) +
        ((df['follow_post_ratio'] > 20) & (df['posts'] < 10)).astype(int) +
        (df['description length'] == 0).astype(int)
    )
    df['SuspicionLevel'] = pd.cut(df['SuspicionScore'], bins=[-1, 2, 5, 8], labels=['Low', 'Medium', 'High'])
    return df

train=add_features(train)
test=add_features(test)

In [9]:
print(train.columns)
print(test.columns)
test.head()

Index(['profile pic', 'nums/length username', 'fullname words',
       'nums/length fullname', 'name==username', 'description length',
       'external URL', 'private', 'posts', 'followers', 'follows', 'fake',
       'follower_follow_ratio', 'follow_post_ratio', 'SuspicionScore',
       'SuspicionLevel'],
      dtype='object')
Index(['profile pic', 'nums/length username', 'fullname words',
       'nums/length fullname', 'name==username', 'description length',
       'external URL', 'private', 'posts', 'followers', 'follows', 'fake',
       'follower_follow_ratio', 'follow_post_ratio', 'SuspicionScore',
       'SuspicionLevel'],
      dtype='object')


,profile pic,nums/length username,fullname words,nums/length fullname,name==username,description length,external URL,private,posts,followers,follows,fake,follower_follow_ratio,follow_post_ratio,SuspicionScore,SuspicionLevel
0,True,0.33,1,0.33,True,30,False,True,35,488,604,0,0.806612,16.777778,2,Low
1,True,0.00,5,0.00,False,64,False,True,3,35,6,0,5.000000,1.500000,2,Low
2,True,0.00,2,0.00,False,82,False,True,319,328,668,0,0.490284,2.087500,1,Low
3,True,0.00,1,0.00,False,143,False,True,273,14890,7369,0,2.020353,26.894161,1,Low
4,True,0.50,1,0.00,False,76,False,True,6,225,356,0,0.630252,50.857143,2,Low


In [10]:
numeric_col=[
    'nums/length username', 'fullname words', 'nums/length fullname','description length', 'posts', 'followers', 'follows','follower_follow_ratio', 'follow_post_ratio', 'SuspicionScore'
]
categorical_cols = ['profile pic', 'external URL', 'name==username', 'private', 'SuspicionLevel']

x_train=train.drop(columns=['fake'])
y_train=train['fake']
x_test=test.drop(columns=['fake'])
y_test=test['fake']

print(x_train.columns)
print(x_test.columns)

Index(['profile pic', 'nums/length username', 'fullname words',
       'nums/length fullname', 'name==username', 'description length',
       'external URL', 'private', 'posts', 'followers', 'follows',
       'follower_follow_ratio', 'follow_post_ratio', 'SuspicionScore',
       'SuspicionLevel'],
      dtype='object')
Index(['profile pic', 'nums/length username', 'fullname words',
       'nums/length fullname', 'name==username', 'description length',
       'external URL', 'private', 'posts', 'followers', 'follows',
       'follower_follow_ratio', 'follow_post_ratio', 'SuspicionScore',
       'SuspicionLevel'],
      dtype='object')


In [11]:
pt=PowerTransformer(method='yeo-johnson',standardize=True)
x_train[numeric_col]=pt.fit_transform(x_train[numeric_col])
x_test[numeric_col]=pt.transform(x_test[numeric_col])

x_train=pd.get_dummies(x_train,columns=categorical_cols,drop_first=True)
x_test=pd.get_dummies(x_test,columns=categorical_cols,drop_first=True)

x_test=x_test.reindex(columns=x_train.columns,fill_value=0)

le=LabelEncoder()
y_train=le.fit_transform(y_train)
y_test=le.transform(y_test)

In [12]:
print(x_train.head(3))


   nums/length username  fullname words  nums/length fullname  \
0              0.918308       -2.108782             -0.335234   
1             -0.899665        0.704495             -0.335234   
2              0.038100        0.704495             -0.335234   

   description length     posts  followers   follows  follower_follow_ratio  \
0            1.277221  0.519301   0.759862  1.049916               0.095545   
1            1.229119  1.357959   1.103279  0.625962               1.370264   
2           -0.849152  0.136798   0.057952 -0.426091               0.478024   

   follow_post_ratio  SuspicionScore  profile pic_True  external URL_True  \
0           0.495814       -0.848587              True              False   
1          -1.126597       -0.848587              True              False   
2          -0.354427       -0.194239              True              False   

   name==username_True  private_True  SuspicionLevel_Medium  \
0                False         False              

In [13]:
print(x_test.head(3))

   nums/length username  fullname words  nums/length fullname  \
0              1.108745       -0.359069              2.995952   
1             -0.899665        2.594054             -0.335234   
2             -0.899665        0.704495             -0.335234   

   description length     posts  followers   follows  follower_follow_ratio  \
0            1.121261  0.556425   0.497977  0.714006              -0.131562   
1            1.323242 -0.480376  -0.595322 -1.658255               1.354353   
2            1.379750  1.395830   0.346429  0.786034              -0.529898   

   follow_post_ratio  SuspicionScore  profile pic_True  external URL_True  \
0           0.177293       -0.194239              True              False   
1          -1.234510       -0.194239              True              False   
2          -1.064787       -0.848587              True              False   

   name==username_True  private_True  SuspicionLevel_Medium  \
0                 True          True              

In [14]:
logreg=LogisticRegression(C=1,penalty='l1',solver='liblinear',max_iter=500)
rf = RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_leaf=6, min_samples_split=2, max_features=0.5)
svc=SVC(C=10,kernel='linear',degree=2,gamma='scale',probability=True)
xgb = XGBClassifier(learning_rate=0.01, max_depth=5, n_estimators=200, subsample=0.8,
                    colsample_bytree=1.0, gamma=0.1, min_child_weight=1,
                    use_label_encoder=False, eval_metric='logloss')
knn = KNeighborsClassifier(n_neighbors=9, metric='manhattan', weights='distance', leaf_size=10)

voting_clf=VotingClassifier(
    estimators=[('logreg',logreg),('rf',rf),('svc',svc),('xgb',xgb),('knn',knn)],
    voting='soft',
    weights=[2,3,2,4,2],
    n_jobs=-1
)
voting_clf.fit(x_train,y_train)

,estimators,"[('logreg', ...), ('rf', ...), ...]"
,voting,'soft'
,weights,"[2, 3, ...]"
,n_jobs,-1
,flatten_transform,True
,verbose,False
,penalty,'l1'
,dual,False
,tol,0.0001
,C,1
,fit_intercept,True


In [15]:


y_pred = voting_clf.predict(x_test)
report = classification_report(y_test, y_pred, output_dict=True)


metrics_df = pd.DataFrame(report).transpose().round(2)
metrics_df = metrics_df.loc[['0', '1', 'accuracy', 'macro avg', 'weighted avg']]
metrics_df.rename(index={'0': 'Class 0 (Real)', '1': 'Class 1 (Fake)'}, inplace=True)
print(metrics_df)


                precision  recall  f1-score  support
Class 0 (Real)       0.98    0.90      0.94    60.00
Class 1 (Fake)       0.91    0.98      0.94    60.00
accuracy             0.94    0.94      0.94     0.94
macro avg            0.94    0.94      0.94   120.00
weighted avg         0.94    0.94      0.94   120.00


In [16]:


joblib.dump(voting_clf, 'model_resources/insta_voting_model.pkl')
joblib.dump(pt, 'model_resources/power_transformer.pkl')
joblib.dump(le, 'model_resources/label_encoder.pkl')
metrics_df.to_csv('model_resources/model_metrics.csv', index=True)
